In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Classroom', 'DRAWING', 'NSSC SIGN', 'Colab Notebooks', 'shravani (6) (1).pdf', 'ai-question bank.gdoc', '755212687_adgxxxxx9b_A1.pdf', 'Untitled document (6).gdoc', 'Visual computing in AI ML-project abstarct.gdoc', 'Untitled document (5).gdoc', 'shravani (6).pdf', 'train', 'train_masks', 'Document_1762848385549.pdf', 'Untitled form.gform', 'Untitled form (Responses).gsheet', 'Startup - Proposal.gdoc', 'Untitled document (4).gdoc', 'Face2Feel report.gdoc', 'Untitled document (3).gdoc', 'report module 1.gdoc', 'shravani (11).pdf', 'shravani (12) (4).pdf', 'shravani (12) (3).pdf', 'shravani (12) (2).pdf', 'shravani (12) (1).pdf', 'shravani (12).pdf', 'Untitled document (2).gdoc', 'Untitled document (1).gdoc', 'Untitled document.gdoc', 'LabProjectGuidelines-2026.pdf', 'datasets', 't-shirt payment.jpeg', 'Assignment1_23CS30051.zip', 'text_sql', 'Screenshot_2026-03-11-11-42-36-64_49b96b5fbae0d12a18edc4a3afe0dfd9.jpg', 'CasDocument.pdf', 'congestion.gdoc', 'gc_dataset.json']


In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer
)

from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "google/flan-t5-base"

DATA_PATH = "text_sql/spider_clean_7000.json"

# =========================
# LOAD DATASET
# =========================

with open(DATA_PATH) as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

# =========================
# LOAD TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT = 512
MAX_OUTPUT = 128


def preprocess(example):

    input_text = example["instruction"] + "\n" + example["input"]

    model_inputs = tokenizer(
        input_text,
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["output"],
        max_length=MAX_OUTPUT,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


dataset = dataset.map(preprocess)

# =========================
# LOAD MODEL
# =========================

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# =========================
# APPLY LORA
# =========================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# =========================
# TRAINING SETTINGS
# =========================

training_args = TrainingArguments(
    output_dir="flan_spider_lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=3e-4,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

# =========================
# TRAIN
# =========================

trainer.train()

# =========================
# SAVE MODEL
# =========================

model.save_pretrained("flan_spider_lora")
tokenizer.save_pretrained("flan_spider_lora")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096


Step,Training Loss
50,42.517432
100,35.003809
150,27.642888
200,24.916372
250,24.227935
300,23.483723
350,23.108748
400,22.824656
450,22.647891
500,22.500112


('flan_spider_lora/tokenizer_config.json', 'flan_spider_lora/tokenizer.json')

In [ ]:
!pip uninstall -y datasets
!pip install datasets --upgrade

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0


In [ ]:
!pip install -q accelerate==0.26.0
!pip install -q transformers==4.36.0
!pip install -q datasets==2.16.0
!pip install -q peft==0.7.0
!pip install -q scipy

print("✅ Packages installed")

✅ Packages installed


In [ ]:
# =========================
# CELL 1: INSTALL LATEST VERSIONS
# =========================
!pip uninstall -y transformers accelerate
!pip install -q transformers>=4.37.0
!pip install -q accelerate>=0.27.0
!pip install -q datasets peft scipy

print("✅ Latest packages installed")

# =========================
# CELL 2: RESTART RUNTIME (IMPORTANT!)
# =========================
# After running Cell 1, go to Runtime → Restart runtime
# Then run Cell 3 below

Found existing installation: transformers 4.36.0
Uninstalling transformers-4.36.0:
  Successfully uninstalled transformers-4.36.0
Found existing installation: accelerate 0.26.0
Uninstalling accelerate-0.26.0:
  Successfully uninstalled accelerate-0.26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.7.0 requires accelerate>=0.21.0, which is not installed.
✅ Latest packages installed


In [ ]:
# =========================
# CELL 1: INSTALL (Run this first)
# =========================
!pip uninstall -y bitsandbytes  # Remove problematic package
!pip install -q transformers>=4.37.0 accelerate datasets peft scipy

print("✅ Packages installed")
print("🔴 NOW: Go to Runtime → Restart runtime, then run Cell 2")

✅ Packages installed
🔴 NOW: Go to Runtime → Restart runtime, then run Cell 2


In [ ]:
# =========================
# CELL 2: FINAL WORKING VERSION
# =========================
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model
import warnings
warnings.filterwarnings('ignore')

# Verify no bitsandbytes is loaded
import sys
if 'bitsandbytes' in sys.modules:
    print("⚠️ bitsandbytes is still loaded! Restart runtime and try again.")
else:
    print("✅ bitsandbytes not loaded - good!")

# Configuration
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATA_PATH = "text_sql/spider_clean_7000.json"
OUTPUT_DIR = "qwen_spider_lora"

# Load dataset
with open(DATA_PATH) as f:
    data = json.load(f)
print(f"✅ Loaded {len(data)} examples")

# Use 500 examples
data = data[:500]
print(f"Using {len(data)} examples for training")

# Format with schema
def format_with_schema(example):
    return {
        "text": f"""<|im_start|>user
{example['instruction']}

{example['input']}
<|im_end|>
<|im_start|>assistant
{example['output']}
<|im_end|>"""
    }

# Apply formatting
formatted_data = [format_with_schema(ex) for ex in data]
dataset = Dataset.from_list(formatted_data)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )

# Tokenize dataset
dataset = dataset.map(tokenize_function, remove_columns=["text"])
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Training examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['test'])}")

# Load model
print("Loading model... (this will take 1-2 minutes)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
print("✅ Model loaded successfully!")

# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)
print("\n📊 Trainable parameters:")
model.print_trainable_parameters()

# ===== FINAL FIXED TRAINING ARGUMENTS =====
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    eval_strategy="epoch",  # Now matches save_strategy
    save_strategy="epoch",  # Both set to "epoch"
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    remove_unused_columns=False,
    optim="adamw_torch",
    warmup_ratio=0.03,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# TRAIN!
print("\n" + "="*50)
print("🚀 STARTING TRAINING")
print("="*50)
print("Expected loss progression:")
print("Epoch 1: 3.5-4.0 → 2.0-2.5")
print("Epoch 2: 2.0-2.5 → 1.2-1.8")
print("="*50)

trainer.train()

# Save model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

# Test the model
print("\n" + "="*50)
print("🧪 TESTING THE MODEL")
print("="*50)

test_queries = [
    {
        "input": """Schema: accounts(account_id, user_id, balance, account_type)
Question: Show me all accounts with balance less than 1000"""
    },
    {
        "input": """Schema: transactions(id, from_account, to_account, amount, timestamp)
Question: Find all transactions from yesterday"""
    }
]

for i, test in enumerate(test_queries):
    print(f"\nTest {i+1}:")
    print(f"Question: {test['input'][:100]}...")

    prompt = f"""<|im_start|>user
Convert this to SQL:

{test['input']}
<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=False,
            num_beams=2
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sql = response.split("<|im_start|>assistant")[-1].strip()
    print(f"Generated SQL: {sql}")
    print("-" * 30)

✅ bitsandbytes not loaded - good!
✅ Loaded 7000 examples
Using 500 examples for training


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Training examples: 450
Validation examples: 50
Loading model... (this will take 1-2 minutes)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Model loaded successfully!


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



📊 Trainable parameters:
trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410

🚀 STARTING TRAINING
Expected loss progression:
Epoch 1: 3.5-4.0 → 2.0-2.5
Epoch 2: 2.0-2.5 → 1.2-1.8


Epoch,Training Loss,Validation Loss
1,0.261591,0.288926
2,0.207953,0.249444


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Model saved to qwen_spider_lora

🧪 TESTING THE MODEL

Test 1:
Question: Schema: accounts(account_id, user_id, balance, account_type)
Question: Show me all accounts with bal...
Generated SQL: user
Convert this to SQL:

Schema: accounts(account_id, user_id, balance, account_type)
Question: Show me all accounts with balance less than 1000

assistant
SELECT * FROM accounts WHERE balance  <  1000

SELECT account_id ,  user_id ,  balance ,  account_type FROM accounts WHERE balance  <  1000

SELECT account_id ,  user_id ,  balance ,  account_type FROM accounts WHERE balance  <  1000

SELECT account_id ,  user_id ,  balance ,  account_type FROM accounts WHERE balance  <  1000
------------------------------

Test 2:
Question: Schema: transactions(id, from_account, to_account, amount, timestamp)
Question: Find all transaction...
Generated SQL: user
Convert this to SQL:

Schema: transactions(id, from_account, to_account, amount, timestamp)
Question: Find all transactions from yesterday

assista

In [ ]:
# =========================
# CELL 1: Train on Spider and Save to Google Drive
# =========================
import json
import torch
import os
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model
import warnings
warnings.filterwarnings('ignore')



# Configuration
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATA_PATH = "text_sql/spider_clean_7000.json"
OUTPUT_DIR = "text_sql/qwen_spider_lora"  # Save directly to Drive

# Load dataset
with open(DATA_PATH) as f:
    data = json.load(f)
print(f"✅ Loaded {len(data)} examples")

data = data[:7000]
print(f"Using {len(data)} examples for training")

# Format with schema
def format_with_schema(example):
    return {
        "text": f"""<|im_start|>user
{example['instruction']}

{example['input']}
<|im_end|>
<|im_start|>assistant
{example['output']}
<|im_end|>"""
    }

# Apply formatting
formatted_data = [format_with_schema(ex) for ex in data]
dataset = Dataset.from_list(formatted_data)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )

# Tokenize dataset
dataset = dataset.map(tokenize_function, remove_columns=["text"])
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Training examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['test'])}")

# Load model
print("Loading model... (this will take 1-2 minutes)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
print("✅ Model loaded successfully!")

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)
print("\n📊 Trainable parameters:")
model.print_trainable_parameters()

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    remove_unused_columns=False,
    optim="adamw_torch",
    warmup_ratio=0.03,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# TRAIN!
print("\n" + "="*50)
print("🚀 STARTING TRAINING")
print("="*50)
trainer.train()

# Save model to Drive
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved permanently to: {OUTPUT_DIR}")



✅ Loaded 7000 examples
Using 7000 examples for training


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

✅ Model loaded successfully!


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



📊 Trainable parameters:
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

🚀 STARTING TRAINING


Epoch,Training Loss,Validation Loss
1,0.212847,0.239200
2,0.203902,0.213088
3,0.161079,0.200081
4,0.146428,0.194627
5,0.124497,0.195361


✅ Model saved permanently to: /content/drive/MyDrive/text_sql/qwen_spider_lora


In [ ]:
# =========================
# FINAL WORKING TEST CODE with Clean Output
# =========================
print("\n" + "="*60)
print("🏦 TESTING SPIDER MODEL ON BANKING SCHEMA")
print("="*60)

complex_test_queries = [
    {
        "category": "EXISTS",
        "input": f"""Schema:
users(user_id, full_name, email, password_hash, user_role, created_at)
accounts(account_id, user_id, account_type, balance, account_no, status, created_at)
transactions(transaction_id, account_id, transaction_type, amount, device_id, transaction_time, status)

Question:
Find users who have never made any transaction"""
    },
    {
        "category": "Multi-table JOIN",
        "input": f"""Schema:
users(user_id, full_name, email)
accounts(account_id, user_id, account_type, balance)
transactions(transaction_id, account_id, amount, transaction_time)
devices(device_id, user_id, device_type, device_os)
ipaddresses(ip_id, ip_address, country, city)

Question:
Show all transactions with user details, device info, and IP location where amount is greater than 50000"""
    },
    {
        "category": "GROUP BY with HAVING",
        "input": f"""Schema:
users(user_id, full_name, email)
transactions(transaction_id, account_id, amount, transaction_time)
accounts(account_id, user_id)

Question:
Show users who have made more than 5 transactions in the last 30 days"""
    }
]

print("\n" + "="*60)
print("🧪 RUNNING COMPLEX QUERY TESTS")
print("="*60)

for i, test in enumerate(complex_test_queries):
    print(f"\n{'='*50}")
    print(f"Test {i+1}: {test['category']}")
    print(f"{'='*50}")
    print(f"Question: {test['input'].split('Question:')[-1].strip()}")

    prompt = f"""<|im_start|>user
{test['input']}
<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=False,
            num_beams=4
        )

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # CLEAN EXTRACTION - Get only the SQL part
    if "<|im_start|>assistant" in full_response:
        sql_part = full_response.split("<|im_start|>assistant")[-1].strip()
    else:
        sql_part = full_response.strip()

    # Remove any "user" or "Schema:" text
    lines = sql_part.split('\n')
    clean_lines = []
    capture = False

    for line in lines:
        # Start capturing when we see SQL keywords
        if any(keyword in line.lower() for keyword in ['select', 'from', 'where', 'join']):
            capture = True
        if capture and line.strip() and not line.strip().startswith('user'):
            clean_lines.append(line)
        elif capture and not line.strip():
            break

    if clean_lines:
        final_sql = '\n'.join(clean_lines)
    else:
        # Fallback: take everything after assistant
        final_sql = sql_part

    print(f"\n📝 GENERATED SQL:")
    print(f"{final_sql}")

    # Basic validation
    has_select = 'select' in final_sql.lower()
    has_from = 'from' in final_sql.lower()

    print(f"\n📊 Valid SQL: {'✅' if has_select and has_from else '❌'}")
    print(f"\n{'-'*50}")


🏦 TESTING SPIDER MODEL ON BANKING SCHEMA

🧪 RUNNING COMPLEX QUERY TESTS

Test 1: EXISTS
Question: Find users who have never made any transaction

📝 GENERATED SQL:
SELECT full_name FROM users WHERE user_id NOT IN (SELECT user_id FROM transactions)

📊 Valid SQL: ✅

--------------------------------------------------

Test 2: Multi-table JOIN
Question: Show all transactions with user details, device info, and IP location where amount is greater than 50000

📝 GENERATED SQL:
Show all transactions with user details, device info, and IP location where amount is greater than 50000

📊 Valid SQL: ❌

--------------------------------------------------

Test 3: GROUP BY with HAVING
Question: Show users who have made more than 5 transactions in the last 30 days

📝 GENERATED SQL:
SELECT T1.full_name FROM users AS T1 JOIN transactions AS T2 ON T1.user_id  =  T2.user_id WHERE transaction_time  >  NOW() - INTERVAL 30 DAY GROUP BY T1.user_id HAVING count(*)  >  5;
```

📊 Valid SQL: ✅

-------------------

In [ ]:
# =========================
# TEST WITH SPIDER'S EXACT FORMAT
# =========================
print("\n" + "="*60)
print("🔗 TESTING WITH SPIDER'S EXACT FORMAT")
print("="*60)

# Use Spider's exact table naming style (singular, lowercase)
spider_style_queries = [
    {
        "category": "Basic INNER JOIN",
        "input": """Schema:
user(user_id, full_name, email, user_role)
account(account_id, user_id, account_type, balance, status)

Question:
Show all users with their account details"""
    },
    {
        "category": "JOIN with WHERE",
        "input": """Schema:
user(user_id, full_name, email)
account(account_id, user_id, balance, account_type)

Question:
Find users who have savings account with balance greater than 10000"""
    },
    {
        "category": "Three-table JOIN",
        "input": """Schema:
user(user_id, full_name, email)
account(account_id, user_id, balance)
transaction(transaction_id, account_id, amount, transaction_time)

Question:
Show user name, account balance, and their transactions"""
    }
]

for i, test in enumerate(spider_style_queries):
    print(f"\n{'='*50}")
    print(f"Test {i+1}: {test['category']}")
    print(f"{'='*50}")

    prompt = f"""<|im_start|>user
{test['input']}
<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=False,
            num_beams=4
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "<|im_start|>assistant" in response:
        sql = response.split("<|im_start|>assistant")[-1].strip()
    else:
        sql = response.strip()

    print(f"\n📝 GENERATED SQL:")
    print(f"{sql}")
    print(f"\n{'-'*50}")


🔗 TESTING WITH SPIDER'S EXACT FORMAT

Test 1: Basic INNER JOIN

📝 GENERATED SQL:
user
Schema:
user(user_id, full_name, email, user_role)
account(account_id, user_id, account_type, balance, status)

Question:
Show all users with their account details

assistant
SELECT T1.full_name ,  T2.account_id ,  T2.account_type ,  T2.balance ,  T2.status FROM user AS T1 JOIN account AS T2 ON T1.user_id  =  T2.user_id
```

--------------------------------------------------

Test 2: JOIN with WHERE

📝 GENERATED SQL:
user
Schema:
user(user_id, full_name, email)
account(account_id, user_id, balance, account_type)

Question:
Find users who have savings account with balance greater than 10000

assistant
SELECT T1.full_name FROM user AS T1 JOIN account AS T2 ON T1.user_id  =  T2.user_id WHERE T2.account_type  =  'savings' AND T2.balance  >  10000
```

--------------------------------------------------

Test 3: Three-table JOIN

📝 GENERATED SQL:
user
Schema:
user(user_id, full_name, email)
account(account

In [ ]:
# =========================
# SAVE CURRENT MODEL FROM DRIVE WITH NEW NAME
# =========================
import os
import shutil


# Define paths
SOURCE_PATH = "/content/drive/MyDrive/text_sql/qwen_spider_lora"  # Your existing model in Drive
CUSTOM_NAME = "upscaled_spider_v2"  # New name for current version
DEST_PATH = f"/content/drive/MyDrive/text_sql/{CUSTOM_NAME}"

# Check if source exists
if os.path.exists(SOURCE_PATH):
    print(f"📁 Found existing model at: {SOURCE_PATH}")
    print(f"📁 Copying to new location: {DEST_PATH}")

    # Copy to new location
    !cp -r "{SOURCE_PATH}" "{DEST_PATH}"

    # Verify copy
    if os.path.exists(DEST_PATH):
        print(f"\n✅ Model successfully saved as '{CUSTOM_NAME}'")
        print(f"\n📂 Files in new model folder:")
        !ls -la {DEST_PATH}

        # Create timestamped backup
        import datetime
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        BACKUP_PATH = f"/content/drive/MyDrive/text_sql/{CUSTOM_NAME}_backup_{timestamp}"
        !cp -r "{SOURCE_PATH}" "{BACKUP_PATH}"
        print(f"\n📦 Backup also saved to: {BACKUP_PATH}")

        # Create zip
        !zip -r "/content/{CUSTOM_NAME}.zip" "{SOURCE_PATH}"
        !cp "/content/{CUSTOM_NAME}.zip" "text_sql/"
        print(f"\n🗜️ Zip file saved to: text_sql/{CUSTOM_NAME}.zip")

        # Show size
        print(f"\n📦 Model size:")
        !du -sh {DEST_PATH}
    else:
        print("❌ Failed to copy model")
else:
    print(f"❌ Source model not found at: {SOURCE_PATH}")
    print("\n🔍 Searching for model in Drive...")
    !ls -la /content/drive/MyDrive/text_sql/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Found existing model at: /content/drive/MyDrive/text_sql/qwen_spider_lora
📁 Copying to new location: /content/drive/MyDrive/text_sql/upscaled_spider_v2

✅ Model successfully saved as 'upscaled_spider_v2'

📂 Files in new model folder:
total 28226
-rw------- 1 root root     1013 Mar 16 10:35 adapter_config.json
-rw------- 1 root root 17462432 Mar 16 10:35 adapter_model.safetensors
-rw------- 1 root root     2507 Mar 16 10:35 chat_template.jinja
drwx------ 2 root root     4096 Mar 16 10:35 checkpoint-6300
drwx------ 2 root root     4096 Mar 16 10:35 checkpoint-7875
-rw------- 1 root root     5218 Mar 16 10:35 README.md
-rw------- 1 root root      661 Mar 16 10:35 tokenizer_config.json
-rw------- 1 root root 11421990 Mar 16 10:35 tokenizer.json

📦 Backup also saved to: /content/drive/MyDrive/text_sql/upscaled_spider_v2_backup_20260316_103514
  adding: content/d

In [ ]:
# =========================
# DOWNLOAD MODEL DIRECTLY TO PC
# =========================

import os
import zipfile

# Define your model path
MODEL_PATH = "text_sql/upscaled_spider_v2"  # or your model name

  adding: content/drive/MyDrive/text_sql/upscaled_spider_v2/ (stored 0%)
  adding: content/drive/MyDrive/text_sql/upscaled_spider_v2/README.md (deflated 65%)
  adding: content/drive/MyDrive/text_sql/upscaled_spider_v2/adapter_model.safetensors (deflated 7%)
  adding: content/drive/MyDrive/text_sql/upscaled_spider_v2/adapter_config.json (deflated 57%)
  adding: content/drive/MyDrive/text_sql/upscaled_spider_v2/chat_template.jinja (deflated 71%)
  adding: content/drive/MyDrive/text_sql/upscaled_spider_v2/tokenizer_config.json (deflated 60%)
  adding: content/drive/MyDrive/text_sql/upscaled_spider_v2/tokenizer.json (deflated 81%)
  adding: content/drive/MyDrive/text_sql/upscaled_spider_v2/checkpoint-6300/ (stored 0%)
  adding: content/drive/MyDrive/text_sql/upscaled_spider_v2/checkpoint-6300/README.md (deflated 65%)
  adding: content/drive/MyDrive/text_sql/upscaled_spider_v2/checkpoint-6300/adapter_model.safetensors (deflated 7%)
  adding: content/drive/MyDrive/text_sql/upscaled_spider_v2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started! Check your Downloads folder.


In [ ]:
# =========================
# LOAD SAVED MODEL + CONTINUE FINETUNING
# =========================

import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import PeftModel
import os

# Paths
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
LORA_PATH = "text_sql/upscaled_spider_v2"
GC_DATA_PATH = "gc_dataset.json"
OUTPUT_DIR = "text_sql/final_text2sql_model"

# =========================
# LOAD TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# =========================
# LOAD BASE MODEL + LORA
# =========================
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(
    base_model,
    LORA_PATH,
    is_trainable=True   # ✅ FIX
)
# ✅ VERY IMPORTANT FIXES
model.train()
model.enable_input_require_grads()

print("✅ Loaded previous fine-tuned model")

# Verify
model.print_trainable_parameters()

# =========================
# LOAD GC DATASET
# =========================
with open(GC_DATA_PATH) as f:
    gc_data = json.load(f)

print(f"Loaded {len(gc_data)} GC samples")

# =========================
# FORMAT DATA (VERY IMPORTANT)
# =========================
def format_data(example):
    return {
        "text": f"""<|im_start|>user
{example['instruction']}

{example['input']}
<|im_end|>
<|im_start|>assistant
{example['output']}
<|im_end|>"""
    }

formatted = [format_data(x) for x in gc_data]
dataset = Dataset.from_list(formatted)

# =========================
# TOKENIZE
# =========================
def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False
    )

dataset = dataset.map(tokenize, remove_columns=["text"])
dataset = dataset.train_test_split(test_size=0.1)

# =========================
# TRAINING CONFIG (LOW LR!!)
# =========================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=5e-5,  # LOWER than before (important!)
    fp16=True,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

# =========================
# TRAINER
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# =========================
# TRAIN
# =========================
print("🚀 Starting second-stage fine-tuning...")
trainer.train()

# =========================
# SAVE FINAL MODEL
# =========================
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ FINAL MODEL SAVED AT: {OUTPUT_DIR}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Loaded previous fine-tuned model
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
Loaded 479 GC samples


Map:   0%|          | 0/479 [00:00<?, ? examples/s]

🚀 Starting second-stage fine-tuning...


Epoch,Training Loss,Validation Loss
1,0.089534,0.091042
2,0.064272,0.079167
3,0.063101,0.076509


✅ FINAL MODEL SAVED AT: /content/drive/MyDrive/text_sql/final_text2sql_model


In [ ]:
# =========================
# LOAD FINAL MODEL
# =========================
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

MODEL_PATH = "text_sql/final_text2sql_model"
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, MODEL_PATH)

# =========================
# GENERATE SQL
# =========================
def generate_sql(question, schema):
    prompt = f"""<|im_start|>user
Convert natural language to SQL using the database schema.

Schema:
{schema}

Question:
{question}
<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.1,   # IMPORTANT: low temp for accuracy
        do_sample=False
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return result.split("assistant")[-1].strip()

# =========================
# TEST
# =========================
schema = """channel_wise_publishing_1(Channels, Facebook, Instagram, Reels, Shorts, Youtube, Total_Count)"""

question = "Find total publishing across all platforms for each channel."

print(generate_sql(question, schema))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SELECT channels, SUM(total_count) AS total_publishing FROM channel_wise_publishing_1 GROUP BY channels;
```


In [ ]:
# =========================
# LOAD MODEL
# =========================
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
import re

MODEL_PATH = "text_sql/final_text2sql_model"
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model.eval()

print("✅ Model loaded\n")

# =========================
# GENERATE FUNCTION
# =========================
def generate_sql(question, schema):
    prompt = f"""<|im_start|>user
Convert natural language to SQL using the database schema.

IMPORTANT:
- Output ONLY SQL query
- No explanation

Schema:
{schema}

Question:
{question}
<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

# =========================
# CLEAN OUTPUT
# =========================
def clean_sql(output):
    output = re.sub(r"```.*?```", "", output, flags=re.DOTALL)
    output = output.split("Note:")[0]
    if ";" in output:
        output = output.split(";")[0] + ";"
    return output.strip()

# =========================
# SCHEMA
# =========================
schema = """channel_wise_publishing_1(Channels, Facebook, Instagram, Reels, Shorts, Youtube, Total_Count)
combined_data2025_3_1_2026_2_28_by_user_1(User, Uploaded_Count, Created_Count, Published_Count, Uploaded_Duration, Created_Duration, Published_Duration)
combined_data2025_3_1_2026_2_28_by_input_type_1(Input_Type, Uploaded_Count, Created_Count, Published_Count, Uploaded_Duration, Created_Duration, Published_Duration)
combined_data2025_3_1_2026_2_28_by_channel_and_user_1(Channel, User, Uploaded_Count, Created_Count, Published_Count, Uploaded_Duration, Created_Duration, Published_Duration)
channel_wise_publishing_duration(Channels, Facebook_Duration, Instagram_Duration, Reels_Duration, Shorts_Duration, Youtube_Duration, Total_Duration)
client_1_combined_data2025_3_1_2026_2_28_1(Channel, Uploaded_Count, Created_Count, Published_Count, Uploaded_Duration, Created_Duration, Published_Duration)
month_wise_duration_1(Month, Total_Uploaded_Duration, Total_Created_Duration, Total_Published_Duration)
combined_data2025_3_1_2026_2_28_by_output_type_1(Output_Type, Uploaded_Count, Created_Count, Published_Count, Uploaded_Duration, Created_Duration, Published_Duration)
highlighted_video_data(Headline, Source, Published, Team_Name, Type, Uploaded_By, Video_ID)
monthly_chart_1(Month, Total_Uploaded, Total_Created, Total_Published)
combined_data2025_3_1_2026_2_28_by_language_2(Language, Uploaded_Count, Created_Count, Published_Count, Uploaded_Duration, Created_Duration, Published_Duration)
"""

# =========================
# TEST QUESTIONS (BIG SET 🔥)
# =========================
questions = [

# BASIC
"Show all users and their uploaded count",
"List all channels and total count",
"Show all months and total published",

# AGGREGATION
"Find total uploaded videos across all users",
"Find average created duration",
"Find total published duration",

# GROUP BY
"Find total uploads per language",
"Find total created count per input type",
"Find total published count per user",

# ORDER BY
"Top 3 users by uploaded count",
"Which channel has highest total count",
"Top 5 languages by uploaded count",

# CONDITIONS
"Find users who uploaded more than 100 videos",
"Find users with published count less than 50",
"Find videos published by team Marketing",

# MULTI CONDITION
"Find users with uploaded count > 50 and created count > 30",
"Find channels where total count is greater than 500",

# COMPUTED
"Find total publishing across all platforms for each channel",
"Find total duration across all platforms for each channel",

# TIME SERIES
"Show monthly uploaded videos trend",
"Show monthly created duration trend",

# RELATION
"Show uploaded count per user per channel",
"Show published count per user per channel",

# AMBIGUOUS
"Show best performing channel",
"Top performing user",
"Show performance by language",

# EDGE CASES
"Find average likes per video",
"Find revenue per channel",
"Find engagement rate by user"
]

# =========================
# RUN TESTS
# =========================
for i, q in enumerate(questions):
    print(f"\n🧪 Test {i+1}")
    print("Q:", q)

    raw = generate_sql(q, schema)
    cleaned = clean_sql(raw)

    print("SQL:", cleaned)
    print("-" * 60)

Loading model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Model loaded


🧪 Test 1
Q: Show all users and their uploaded count
SQL: user
Convert natural language to SQL using the database schema.

IMPORTANT:
- Output ONLY SQL query
- No explanation

Schema:
channel_wise_publishing_1(Channels, Facebook, Instagram, Reels, Shorts, Youtube, Total_Count)
combined_data2025_3_1_2026_2_28_by_user_1(User, Uploaded_Count, Created_Count, Published_Count, Uploaded_Duration, Created_Duration, Published_Duration)
combined_data2025_3_1_2026_2_28_by_input_type_1(Input_Type, Uploaded_Count, Created_Count, Published_Count, Uploaded_Duration, Created_Duration, Published_Duration)
combined_data2025_3_1_2026_2_28_by_channel_and_user_1(Channel, User, Uploaded_Count, Created_Count, Published_Count, Uploaded_Duration, Created_Duration, Published_Duration)
channel_wise_publishing_duration(Channels, Facebook_Duration, Instagram_Duration, Reels_Duration, Shorts_Duration, Youtube_Duration, Total_Duration)
client_1_combined_data2025_3_1_2026_2_28_1(Channel, Uploaded_Coun

In [ ]:
# =========================
# CELL 2: Load Spider Model from Drive and Fine-tune on Banking
# =========================
import json
import torch
import os
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, PeftModel
import warnings
warnings.filterwarnings('ignore')



# =========================
# CONFIGURATION
# =========================
BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SPIDER_MODEL_PATH = "text_sql/qwen_spider_lora"  # Load from Drive
BANKING_DATA_PATH = "text_sql/banking_dataset.json"  # Save your banking data here
OUTPUT_DIR = "text_sql/qwen_banking_final"


# =========================
# LOAD SPIDER-TRAINED MODEL FROM DRIVE
# =========================
print(f"\n🔄 Loading Spider-trained model from: {SPIDER_MODEL_PATH}")

# Load tokenizer from Drive
tokenizer = AutoTokenizer.from_pretrained(
    SPIDER_MODEL_PATH,
    trust_remote_code=True
)
print("✅ Tokenizer loaded!")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# Load the trained adapter from Drive
model = PeftModel.from_pretrained(
    base_model,
    SPIDER_MODEL_PATH,
    is_trainable=True,
    local_files_only=True
)
print("✅ Spider model loaded successfully from Drive!")

# =========================
# LOAD BANKING DATASET
# =========================
print("\n📂 Loading banking dataset...")
with open(BANKING_DATA_PATH) as f:
    data = json.load(f)
print(f"✅ Loaded {len(data)} banking examples")

# Format with banking schema
def format_banking(example):
    return {
        "text": f"""<|im_start|>user
{example['instruction']}

{example['input']}
<|im_end|>
<|im_start|>assistant
{example['output']}
<|im_end|>"""
    }

formatted_data = [format_banking(ex) for ex in data]
dataset = Dataset.from_list(formatted_data)

# Tokenizer settings
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=1024,
        padding=False,
    )

dataset = dataset.map(tokenize_function, remove_columns=["text"])
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Training examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['test'])}")

# =========================
# TRAINING ARGUMENTS
# =========================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    remove_unused_columns=False,
    optim="adamw_torch",
    warmup_ratio=0.1,
)

# =========================
# TRAINER
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# =========================
# TRAIN!
# =========================
print("\n" + "="*50)
print("🚀 STARTING BANKING FINE-TUNING")
print("="*50)
print("Expected loss progression:")
print("Epoch 1: 0.25 → 0.20")
print("Epoch 2: 0.20 → 0.15")
print("Epoch 3: 0.15 → 0.12")
print("="*50)

trainer.train()

# =========================
# SAVE FINAL MODEL TO DRIVE
# =========================
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Final model saved permanently to: {OUTPUT_DIR}")

# =========================
# TEST THE FINAL MODEL (WITH STOP SEQUENCE)
# =========================
print("\n" + "="*50)
print("🧪 TESTING BANKING MODEL")
print("="*50)

test_banking_queries = [
    {
        "input": """Schema:
users(user_id, full_name, email, phone, role, status)
accounts(account_id, user_id, account_number, account_type, balance, currency)
transactions(transaction_id, sender_account, receiver_account, amount, transaction_time)

Question: Show me all accounts with balance less than 5000"""
    },
    {
        "input": """Schema:
users(user_id, full_name, email)
loginlogs(login_id, user_id, ip_id, login_time, login_status)
ipaddresses(ip_id, ip_address, country, city)

Question: Find all failed login attempts from last 24 hours"""
    }
]

for i, test in enumerate(test_banking_queries):
    print(f"\nTest {i+1}:")
    print(f"Question: {test['input'][:150]}...")

    prompt = f"""<|im_start|>user
Convert this to SQL:

{test['input']}
<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=False,
            num_beams=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # CLEAN UP THE OUTPUT - Take only the first SQL statement
    sql_part = response.split("<|im_start|>assistant")[-1].strip()

    # Remove any comment blocks and repeated content
    if '*/' in sql_part:
        sql_part = sql_part.split('*/')[0].strip()

    # Take only until newline or natural stop
    sql_part = sql_part.split('\n\n')[0]  # Stop at double newline
    sql_part = sql_part.split('*/')[0].strip()  # Stop at comment

    print(f"Generated SQL: {sql_part}")
    print("-" * 30)

# Replace the last two lines with this:

# Create zip directly in your Drive folder
!zip -r /content/drive/MyDrive/text_sql/qwen_banking_final.zip {OUTPUT_DIR}
print(f"✅ Zip file saved to: /content/drive/MyDrive/text_sql/qwen_banking_final.zip")

# Optional: Also keep a local copy for quick download
!cp /content/drive/MyDrive/text_sql/qwen_banking_final.zip /content/



🔄 Loading Spider-trained model from: /content/drive/MyDrive/text_sql/qwen_spider_lora
✅ Tokenizer loaded!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Spider model loaded successfully from Drive!

📂 Loading banking dataset...
✅ Loaded 72 banking examples


Map:   0%|          | 0/72 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training examples: 64
Validation examples: 8

🚀 STARTING BANKING FINE-TUNING
Expected loss progression:
Epoch 1: 0.25 → 0.20
Epoch 2: 0.20 → 0.15
Epoch 3: 0.15 → 0.12


Epoch,Training Loss,Validation Loss
1,1.812781,0.464838
2,0.280576,0.215940
3,0.220494,0.196120


✅ Final model saved permanently to: /content/drive/MyDrive/text_sql/qwen_banking_final

🧪 TESTING BANKING MODEL

Test 1:
Question: Schema:
users(user_id, full_name, email, phone, role, status)
accounts(account_id, user_id, account_number, account_type, balance, currency)
transacti...
Generated SQL: user
Convert this to SQL:
------------------------------

Test 2:
Question: Schema:
users(user_id, full_name, email)
loginlogs(login_id, user_id, ip_id, login_time, login_status)
ipaddresses(ip_id, ip_address, country, city)

...
Generated SQL: user
Convert this to SQL:
------------------------------
updating: content/drive/MyDrive/text_sql/qwen_banking_final/ (stored 0%)
updating: content/drive/MyDrive/text_sql/qwen_banking_final/checkpoint-32/ (stored 0%)
updating: content/drive/MyDrive/text_sql/qwen_banking_final/checkpoint-32/README.md (deflated 65%)
updating: content/drive/MyDrive/text_sql/qwen_banking_final/checkpoint-32/adapter_model.safetensors (deflated 8%)
updating: content/drive/M

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================
# TEST THE FINAL MODEL (FIXED - WORKS!)
# =========================
print("\n" + "="*50)
print("🧪 TESTING BANKING MODEL")
print("="*50)

test_banking_queries = [
    {
        "input": """Schema:
users(user_id, full_name, email, phone, role, status)
accounts(account_id, user_id, account_number, account_type, balance, currency)
transactions(transaction_id, sender_account, receiver_account, amount, transaction_time)

Question: Show me all accounts with balance less than 5000"""
    },
    {
        "input": """Schema:
users(user_id, full_name, email)
loginlogs(login_id, user_id, ip_id, login_time, login_status)
ipaddresses(ip_id, ip_address, country, city)

Question: Find all failed login attempts from last 24 hours"""
    }
]

for i, test in enumerate(test_banking_queries):
    print(f"\n{'='*30}")
    print(f"Test {i+1}:")
    print(f"{'='*30}")
    print(f"QUESTION: {test['input'][:150]}...")

    prompt = f"""<|im_start|>user
Convert this to SQL:

{test['input']}
<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=False,
            num_beams=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # DEBUG - print raw response to see what's happening
    print(f"\n🔍 RAW RESPONSE: {full_response}")

    # Extract just the SQL part - IMPROVED METHOD
    if "<|im_start|>assistant" in full_response:
        sql_part = full_response.split("<|im_start|>assistant")[-1].strip()
    else:
        sql_part = full_response.strip()

    # Remove any "user" or "Convert this to SQL" text
    lines = sql_part.split('\n')
    clean_lines = []
    for line in lines:
        # Skip lines that are just "user" or contain the prompt
        if line.strip() and not line.strip() == "user" and not "Convert this to SQL" in line:
            clean_lines.append(line)
        elif clean_lines and not line.strip():
            break

    if clean_lines:
        final_sql = '\n'.join(clean_lines)
    else:
        final_sql = sql_part

    print(f"\n📝 GENERATED SQL:")
    print(f"{final_sql}")
    print(f"\n{'-'*50}")


🧪 TESTING BANKING MODEL

Test 1:
QUESTION: Schema:
users(user_id, full_name, email, phone, role, status)
accounts(account_id, user_id, account_number, account_type, balance, currency)
transacti...

🔍 RAW RESPONSE: user
Convert this to SQL:

Schema:
users(user_id, full_name, email, phone, role, status)
accounts(account_id, user_id, account_number, account_type, balance, currency)
transactions(transaction_id, sender_account, receiver_account, amount, transaction_time)

Question: Show me all accounts with balance less than 5000

assistant
SELECT * FROM accounts WHERE balance  <  5000
ORDER BY balance DESC
LIMIT 10
*/
SELECT * FROM accounts WHERE balance  <  5000
ORDER BY balance DESC
LIMIT 10
*/
SELECT * FROM accounts WHERE balance  <  5000
ORDER BY balance DESC
LIMIT 10
*/
SELECT * FROM accounts WHERE balance  <  5000
ORDER BY balance DESC
LIMIT 10
*/
SELECT * FROM accounts WHERE balance  <  5000
ORDER BY balance DESC
LIMIT 10
*/
SELECT * FROM accounts WHERE balance  <  5000
ORDER BY ba